# NB-03-2 スピン $j$ の回転と、量子ビットの合成スピン

[角運動量のラダー演算子のノート](../../研究ノート/05_量子力学/angular_momentum_ladder_operators.md)の行列（NB-03-1）を使って、

1. スピン $j$ を回転させたときの、$J_z$ の測定確率（$j=\frac12$ では NB-01 の $\cos^2\frac\theta2$）
2. $2\pi$ 回転が $(-1)^{2j}$ になること（NB-02 の二重被覆の一般化）
3. $n$ 個の量子ビット（スピン $\frac12$）を並べたときの合成スピン（Qiskit の `SparsePauliOp` で作る）

を確かめます。3 は、ノートの範囲の少し先（角運動量の合成）で、Phase 2 の複数量子ビットへの橋渡しです。

**単位**：$\hbar=1$ とします。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from fractions import Fraction
from numpy.typing import NDArray
from plotly.subplots import make_subplots
from scipy.linalg import expm

np.set_printoptions(precision=4, suppress=True)

# 型の別名（型ヒントを読みやすくするため。実行時には何も確かめない）
Matrix = NDArray[np.complex128]   # (2j+1)×(2j+1) の複素行列
Floats = NDArray[np.float64]      # 実数の配列

# 単位：ħ = 1 とする（ノートの式で ħ を 1 に置き換えたもの）
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなく Python のリストを渡す
# （numpy 配列だと、スライダーを動かしたときに plotly が ValueError を出すことがある。NB-02 を参照）


def m_values(j: float) -> Floats:
    """m = j, j−1, …, −j（上から順。2j+1 個）"""
    return np.arange(j, -j - 1e-9, -1.0)


def spin_matrices(j: float) -> dict[str, Matrix]:
    """ノートの Part VI の公式から、角運動量 j の行列を作る（基底 |j,m⟩ を m の大きい順に並べる）"""
    ms = m_values(j)
    d = len(ms)
    Jz = np.diag(ms).astype(complex)
    Jp = np.zeros((d, d), dtype=complex)
    for k in range(1, d):                                   # J+|j,m⟩ = √(j(j+1) − m(m+1)) |j,m+1⟩
        m = ms[k]
        Jp[k - 1, k] = np.sqrt(j * (j + 1) - m * (m + 1))
    Jm = Jp.conj().T                                        # J− = J+†（ノートの Part II）
    Jx = (Jp + Jm) / 2                                      # J± = Jx ± iJy を解いたもの
    Jy = (Jp - Jm) / 2j
    return {"x": Jx, "y": Jy, "z": Jz, "+": Jp, "-": Jm}


def comm(A: Matrix, B: Matrix) -> Matrix:
    """交換子 [A, B] = AB − BA"""
    return A @ B - B @ A


def frac(x: float) -> str:
    """0.5 → 1/2 のように、半整数を分数で表す（表示用）"""
    return str(Fraction(x).limit_denominator(2))

## 1. 回転したスピンを測る

一番上の状態 $|j,j\rangle$（$z$ 軸の向き）を、$y$ 軸のまわりに角度 $\theta$ 回して（$e^{-i\theta J_y}$）から $J_z$ を測ると、$m'$ が出る確率は

$$
P(m')=\binom{2j}{j+m'}\cos^{2(j+m')}\frac\theta2\ \sin^{2(j-m')}\frac\theta2
$$

の二項分布になります（スピン $j$ を、同じ向きのスピン $\frac12$ が $2j$ 個そろったものと見ると、それぞれが確率 $\cos^2\frac\theta2$ で上向きになる、という形です。§3 も参照）。$j=\frac12$ では $P(+\frac12)=\cos^2\frac\theta2$ で、NB-01 の測定確率と一致します。期待値は $\langle J_z\rangle=j\cos\theta$ です。

In [ ]:
from math import comb


def rotated_probabilities(j: float, theta: float) -> Floats:
    """e^{−iθJy}|j,j⟩ で J_z を測ったときの、m' = j, …, −j の確率（行列の指数関数で計算）"""
    psi = expm(-1j * theta * spin_matrices(j)["y"])[:, 0]
    return np.abs(psi) ** 2


def binomial_formula(j: float, theta: float) -> Floats:
    n = int(round(2 * j))
    return np.array([comb(n, int(round(j + m))) * np.cos(theta / 2) ** (2 * (j + m)) * np.sin(theta / 2) ** (2 * (j - m))
                     for m in m_values(j)])


for twice_j in range(1, 11):
    j = twice_j / 2
    for theta in np.linspace(0, 2 * np.pi, 13):
        p = rotated_probabilities(j, theta)
        assert np.allclose(p, binomial_formula(j, theta))
        assert np.isclose(p @ m_values(j), j * np.cos(theta))              # ⟨J_z⟩ = j cos θ
assert np.isclose(rotated_probabilities(0.5, 1.2)[0], np.cos(0.6) ** 2)   # j = 1/2：cos²(θ/2)
print("二項分布の式と ⟨J_z⟩ = j cos θ を、j = 1/2 〜 5、13 個の角度で確認しました")

fig = go.FigureWidget([go.Bar(x=[], y=[], name="行列の指数関数で計算", marker_color="#1f77b4"),
                       go.Scatter(x=[], y=[], mode="markers", name="二項分布の式",
                                  marker=dict(symbol="line-ew", size=30, line=dict(width=3, color="black")))])
fig.update_layout(height=400, width=820, margin=dict(l=40, r=20, t=30, b=40), yaxis=dict(range=[0, 1.05]),
                  xaxis_title="測定値 m'", yaxis_title="確率 P(m')", legend=dict(x=0.65, y=0.98))
j_rot = W.SelectionSlider(options=[(frac(k / 2), k / 2) for k in range(1, 11)], value=2.0, description="j")
theta_rot = W.FloatSlider(value=60, min=0, max=360, step=5, description="θ [度]")
info = W.HTML()


def update(*_: object) -> None:
    """スライダーが動くたびに呼ばれ、確率の棒を描き直す"""
    j, theta = j_rot.value, np.radians(theta_rot.value)
    p = rotated_probabilities(j, theta)
    labels = [frac(m) for m in m_values(j)]
    with fig.batch_update():
        fig.data[0].x, fig.data[0].y = labels, [float(v) for v in p]
        fig.data[1].x, fig.data[1].y = labels, [float(v) for v in binomial_formula(j, theta)]
    info.value = f"⟨J_z⟩ = {float(p @ m_values(j)):+.3f}（= j cos θ = {j * np.cos(theta):+.3f}）"


for w in (j_rot, theta_rot):
    w.observe(update, "value")
update()
W.VBox([W.HBox([j_rot, theta_rot]), info, fig])

## 2. $2\pi$ 回転は $(-1)^{2j}$

$e^{-i\theta J_z}$ は対角行列 $\operatorname{diag}(e^{-i\theta m})$ なので、$\theta=2\pi$ では $e^{-2\pi im}=(-1)^{2m}=(-1)^{2j}$（$m$ と $j$ は整数だけ違う）が並び、

$$
e^{-2\pi iJ_z}=(-1)^{2j}\,I
$$

です。半整数の $j$ では $-I$（NB-02 の二重被覆。$j=\frac12$ がその例）、整数の $j$ では $I$ です。下の図は、回転のトレースを次元で割ったもの $\chi_j(\theta)/(2j+1)$ で、$\chi_j(\theta)=\operatorname{tr}e^{-i\theta J_z}=\dfrac{\sin\frac{(2j+1)\theta}2}{\sin\frac\theta2}$ です。整数の $j$ は周期 $2\pi$、半整数の $j$ は周期 $4\pi$ です。

In [ ]:
for twice_j in range(0, 11):
    j = twice_j / 2
    Jz = spin_matrices(j)["z"]
    assert np.allclose(expm(-2j * np.pi * Jz), (-1) ** twice_j * np.eye(twice_j + 1))
    for t in (0.7, 2.3, 5.1, 9.0):
        assert np.isclose(np.trace(expm(-1j * t * Jz)).real, np.sin((2 * j + 1) * t / 2) / np.sin(t / 2))
print("2π 回転 = (−1)^{2j} I と、トレースの式を確認しました")

thetas = np.linspace(1e-3, 4 * np.pi, 600)
fig_c = go.Figure()
for j, color, dash in [(0.5, "#d62728", "solid"), (1.5, "#e67e00", "dot"), (1.0, "#1f77b4", "solid"), (2.0, "#2ca02c", "dot")]:
    chi = np.sin((2 * j + 1) * thetas / 2) / np.sin(thetas / 2) / (2 * j + 1)
    fig_c.add_trace(go.Scatter(x=(thetas / np.pi).tolist(), y=chi.tolist(), mode="lines", name=f"j = {frac(j)}",
                               line=dict(color=color, dash=dash)))
fig_c.add_vline(x=2, line_dash="dash", line_color="#888", annotation_text="θ = 2π")
fig_c.update_layout(height=380, width=820, margin=dict(l=40, r=20, t=30, b=40),
                    xaxis_title="θ / π", yaxis_title="χ_j(θ) / (2j+1)")
fig_c

## 3. 量子ビットを並べると、大きなスピンになる（Qiskit）

$n$ 個の量子ビットの全スピンを

$$
S_a=\frac12\sum_{k=1}^{n}\sigma_a^{(k)}\qquad(a=x,y,z)
$$

とします（$\sigma_a^{(k)}$ は $k$ 番目の量子ビットだけに作用するパウリ行列。Qiskit の `SparsePauliOp` で作ります）。$S_a$ も角運動量の交換関係を満たすので、$S^2$ の固有値は $j(j+1)$ の形で、$j$ は $\frac n2,\frac n2-1,\dots$ です。それぞれの $j$ が現れる回数は $\binom{n}{n/2-j}-\binom{n}{n/2-j-1}$ です（2個なら $j=1$ が1組と $j=0$ が1組）。

さらに、全員が上向き $|00\cdots0\rangle$（$|0\rangle$ は $\sigma_z=+1$）から $S_-$ で下りていくと、$j=\frac n2$ の梯子ができます。その梯子の上で $S_a$ の行列を作ると、NB-03-1 の `spin_matrices(n/2)` と一致します。これが、§1 で「スピン $j$ は、スピン $\frac12$ が $2j$ 個そろったもの」と言った意味です。

In [ ]:
from collections import Counter

from qiskit.quantum_info import SparsePauliOp, Statevector


def total_spin(n: int) -> dict[str, Matrix]:
    """n 量子ビットの全スピン S_a = ½ Σ_k σ_a^(k) を、SparsePauliOp で作って行列にする"""
    S = {a: SparsePauliOp.from_sparse_list([(a.upper(), [k], 0.5) for k in range(n)], num_qubits=n) for a in "xyz"}
    return {a: op.to_matrix() for a, op in S.items()}


def copies(n: int, j: float) -> int:
    """n 個のスピン ½ の中に、合成スピン j が現れる回数"""
    k = int(round(n / 2 - j))
    return comb(n, k) - (comb(n, k - 1) if k >= 1 else 0)


for n in range(1, 7):
    S = total_spin(n)
    assert np.allclose(comm(S["x"], S["y"]), 1j * S["z"])                     # 角運動量の交換関係
    S2 = S["x"] @ S["x"] + S["y"] @ S["y"] + S["z"] @ S["z"]
    ev = np.linalg.eigvalsh(S2)
    js = Counter(round(float((-1 + np.sqrt(1 + 4 * e)) / 2) * 2) / 2 for e in ev)   # 固有値 j(j+1) から j を逆算
    for j, count in js.items():
        assert count == copies(n, j) * (2 * j + 1)                             # 次元 = 回数 × (2j+1)
    desc = "、".join(f"j = {frac(j)} が {copies(n, j)} 組" for j in sorted(js, reverse=True))
    print(f"n = {n}：{desc}")

In [ ]:
def symmetric_ladder(n: int) -> list[Matrix]:
    """|00…0⟩ から S− で下りてできる、j = n/2 の梯子（規格化した状態の列）"""
    S = total_spin(n)
    Sm = S["x"] - 1j * S["y"]
    v = np.zeros(2**n, dtype=complex)
    v[0] = 1.0                                                                 # |00…0⟩：全員が上向き
    ladder = [v]
    for _ in range(n):
        v = Sm @ v
        v = v / np.linalg.norm(v)
        ladder.append(v)
    return ladder


for n in range(1, 6):
    S = total_spin(n)
    B = np.array(symmetric_ladder(n)).T                                        # 列が梯子の状態
    for a in "xyz":
        assert np.allclose(B.conj().T @ S[a] @ B, spin_matrices(n / 2)[a])     # 梯子の上の行列 = spin_matrices(n/2)
print("|00…0⟩ から S− で下りた梯子の上で、S_a の行列は spin_matrices(n/2) と一致しました（n = 1〜5）\n")

for n in (2, 3):
    print(f"n = {n} の梯子（j = {frac(n / 2)}）：")
    for k, v in enumerate(symmetric_ladder(n)):
        terms = {b: complex(c) for b, c in Statevector(v).to_dict().items() if abs(c) > 1e-12}
        text = " + ".join(f"{c.real:.3f}|{b}⟩" for b, c in terms.items())
        print(f"  m = {frac(n / 2 - k):>4}：{text}")

## 観察のポイント

- 回転したスピンの測定確率は二項分布です。$j$ が大きいほど分布は鋭くなり、$\langle J_z\rangle=j\cos\theta$ のまわりに集まります（古典的なコマの向きに近づく）。
- $2\pi$ 回すと、半整数の $j$ では状態の符号が反転し、整数の $j$ では元に戻ります。NB-02 の二重被覆は、$j=\frac12$ の場合でした。
- 2つの量子ビットの全スピンは、$j=1$（3状態）と $j=0$（1状態）に分かれます。$j=1$ の真ん中の状態 $\frac1{\sqrt2}(|01\rangle+|10\rangle)$ は、もつれた状態です。Phase 2 でベル状態として扱います。

**次に試すこと**：§3 で、$j=0$ の状態（1重項）を探してみましょう。$S^2$ の固有値 $0$ の固有ベクトルを求めると、$\frac1{\sqrt2}(|01\rangle-|10\rangle)$ になるはずです。